In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
from datetime import datetime
import os

# --- Core Logic Functions ---
def categorize_actor(actor_name):
    if pd.isna(actor_name): return 'Unidentified'
    name = str(actor_name).lower()
    if any(x in name for x in ['military forces of myanmar', 'police forces of myanmar', 'state administration council', 'border guard force', 'people\'s militia force']):
        return 'State Forces'
    elif any(x in name for x in ['pyu saw htee', 'thway thauk', 'blood comrades', 'swan arr shin', 'pro-military']):
        return 'Pro-Junta Militia'
    resistance_keywords = ['pdf', 'people\'s defense force', 'local defense force', 'kndf', 'karenni nationalities defense force', 'chinland defense force', 'cdf', 'unidentified anti-coup armed group', 'guerrilla', 'ogre', 'revolution', 'defense force', 'resistance', 'pafd', 'yaw defense force', 'ydf', 'dragon army', 'mrda', 'chindwin attack force', 'myingyan black tiger', 'mbt', 'black eagle', 'underground warriors', 'young force', 'ug-force', 'guerrilla force', 'defense team', 'ddt', 'special task force', 'sstf', 'attack force', 'generation z', 'gen z', 'drone strike', 'mdds', 'dark shadow', 'leopard army', 'blpa', 'chindwin brothers', 'taung nyo', 'eagle force', 'brave heart', 'danger force', 'red bandana', 'phoenix sgg', 'freeland attack', 'fla', 'support organization', 'commando', 'special force', 'vanguard', 'victory force', 'justice force', 'strike force', 'task force', 'people\'s army', 'liberation army', 'bha', 'tpf', 'kpaf', 'mmu', 'kso', 'sgg', 'baf', 'column', 'urban', 'cgm', 'technical team', 'shar htoo waw', 'king cobra', 'defence force', 'defence team', 'militia', 'security force', 'black k', 'thu rain', 'freedom force', 'pdaf', 'galon force', 'federal army', 'tiger force', 'ranger group', 'truth army', 'anonymous', 'oak awe', 'snake eyes']
    if any(x in name for x in resistance_keywords):
        return 'Resistance'
    eao_keywords = ['knu', 'knla', 'kndo', 'kia', 'kio', 'tnla', 'pslf', 'mndaa', 'mntjp', 'aa ', 'ula', 'rcss', 'ssa', 'knpp', 'ka ', 'cnp', 'sspp', 'pnlo', 'sna', 'cnf', 'cna', 'pno', 'pna', 'brotherhood alliance', 'northern alliance', 'three brotherhood', 'absdf', 'knlp', 'kpc', 'dkba', 'mnda', 'alp', 'ala', 'nssaa', 'shanni', 'kachin', 'karen', 'shan state', 'arakan', 'mon state', 'nmsp', 'nmla', 'chin brotherhood', 'rohingya', 'arsa', 'kaw thoo lei', 'pa-oh', 'ta\'ang', 'palaung', 'kokang', 'wa state', 'uwsa', 'mon national', 'naga', 'nscn', 'ktla']
    if any(x in name for x in eao_keywords):
        return 'EAOs'
    if 'protesters' in name: return 'Protesters'
    elif 'rioters' in name: return 'Rioters'
    elif 'civilians' in name: return 'Civilians'
    if 'unidentified' in name: return 'Unidentified'
    else: return 'Other Groups'

def extract_keywords(text_series, top_n=20):
    from collections import Counter
    stopwords = set(['the', 'and', 'a', 'to', 'in', 'of', 'on', 'with', 'for', 'at', 'by', 'from', 'an', 'is', 'was', 'were', 'it', 'as', 'that', 'this', 'reported', 'took', 'place', 'around', 'near', 'village', 'township', 'district', 'region', 'state', 'myanmar', 'burma', 'forces', 'military', 'junta', 'people', 'defence', 'force', 'pdf', 'tatmadaw', 'knu', 'kia', 'pdf', 'ldf', 'eao', 'ia', 'army'])
    all_words = ' '.join(text_series.fillna('').str.lower().replace(r'[^a-zA-Z\s]', '', regex=True)).split()
    filtered_words = [word for word in all_words if word not in stopwords and len(word) > 3]
    counts = Counter(filtered_words).most_common(top_n)
    return pd.DataFrame(counts, columns=['Keyword', 'Frequency'])

def extract_health_impacts(df):
    text_series = df['notes'].fillna('').str.lower()
    event_series = df['sub_event_type'].fillna('').str.lower()
    health_keywords = [r'hospital', r'clinic', r'medical', r'doctor', r'nurse', r'health', r'ambulance', r'medicine', r'patient', r'pharmacy', r'red cross', r'world health organization', r'who-led', r'unicef', r'displacement', r'malnutrition', r'injury', r'wounded', r'healthcare', r'sanitation', r'vaccination', r'epidemic', r'airstrike near hospital', r'shelling near hospital']
    pattern = '|'.join(health_keywords)
    health_hits = text_series.str.contains(pattern, case=False, regex=True)
    wellbeing_events = ['sexual violence', 'arrests', 'abduction', 'looting', 'property destruction']
    event_hits = event_series.apply(lambda x: any(e in x for e in wellbeing_events))
    return health_hits | event_hits


In [2]:
# --- Data Loading & Preprocessing ---
DATA_PATH = '../data/myanmar_conflict_clean.csv'
if not os.path.exists(DATA_PATH):
    print(f'Error: Data file not found at {DATA_PATH}')
else:
    df = pd.read_csv(DATA_PATH)
    df['event_date'] = pd.to_datetime(df['event_date'])
    df = df[df['event_date'] >= '2021-02-01']
    df['year_month'] = df['event_date'].dt.strftime('%Y-%m')
    df['actor1_clean'] = df['actor1'].apply(categorize_actor)
    df['actor2_clean'] = df['actor2'].apply(categorize_actor)

    node_color_map = {
        'State Forces': '#1e293b', 'Pro-Junta Militia': '#991b1b', 'Resistance': '#10b981', 
        'EAOs': '#0369a1', 'Civilians': '#f59e0b', 'Protesters': '#8b5cf6', 
        'Rioters': '#f97316', 'Unidentified': '#475569', 'Other Groups': '#64748b'
    }
    print(f'Data loaded: {len(df)} events.')


Data loaded: 91441 events.


In [3]:
# --- Actor Network Analysis (Impacts & Interaction Network) ---
# 1. Fatality Impact by Actor
a1_impact = df.groupby('actor1_clean')['fatalities'].sum().reset_index().rename(columns={'actor1_clean': 'actor'})
a2_impact = df.groupby('actor2_clean')['fatalities'].sum().reset_index().rename(columns={'actor2_clean': 'actor'})
actor_stats = pd.concat([a1_impact, a2_impact]).groupby('actor')['fatalities'].sum().reset_index()
actor_stats = actor_stats[actor_stats['actor'] != 'Unidentified'].sort_values('fatalities')

fig_bar = px.bar(actor_stats, x='fatalities', y='actor', orientation='h', color='actor', color_discrete_map=node_color_map)
fig_bar.update_layout(title='Fatality Impact by Actor Category', showlegend=False)
fig_bar.show()

# 2. Actor Interaction Network
interactions = df[(df['actor1_clean'] != df['actor2_clean']) & (df['actor2_clean'] != 'Unidentified')]
if not interactions.empty:
    adj = interactions.groupby(['actor1_clean', 'actor2_clean']).agg(interaction_count=('event_id_cnty', 'count'), total_fatalities=('fatalities', 'sum')).reset_index()
    adj = adj[adj['interaction_count'] >= 2]
    
    G = nx.Graph()
    for _, row in adj.iterrows():
        G.add_edge(row['actor1_clean'], row['actor2_clean'], weight=row['interaction_count'])
    
    pos = nx.spring_layout(G, k=0.8, iterations=100, seed=42)
    edge_traces = []
    max_fatalities = adj['total_fatalities'].max()
    
    for _, row in adj.iterrows():
        x0, y0 = pos[row['actor1_clean']]
        x1, y1 = pos[row['actor2_clean']]
        edge_width = 1 + (row['total_fatalities'] / max_fatalities * 10)
        edge_traces.append(go.Scatter(x=[x0, x1, None], y=[y0, y1, None], line=dict(width=edge_width, color='#475569'), opacity=0.6, mode='lines'))

    node_trace = go.Scatter(
        x=[pos[node][0] for node in G.nodes()], y=[pos[node][1] for node in G.nodes()],
        mode='markers+text', text=[node for node in G.nodes()], textposition='top center',
        marker=dict(color=[node_color_map.get(node, '#64748b') for node in G.nodes()], size=20, line_width=2)
    )

    fig_net = go.Figure(data=edge_traces + [node_trace], layout=go.Layout(title='ACTOR INTERACTION NETWORK', showlegend=False, xaxis=dict(showgrid=False, zeroline=False, showticklabels=False), yaxis=dict(showgrid=False, zeroline=False, showticklabels=False), paper_bgcolor='white', plot_bgcolor='white'))
    fig_net.show()
